# Notebook 2 — Build the index and check retrieval

Run the notebook to build or reuse the Chroma index, inspect text, table, and figure chunks, then compare keyword and vector search. The final report checks how often the right document and page appear among the first four results for 16 questions.

For the implementation, look up `elements_to_chunks` and `fuse` in `document_rag.py`.


In [1]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
if not (HERE / "document_rag.py").exists():
    raise SystemExit("Set the notebook's working directory to 16_agentic_document_rag.")
sys.path.insert(0, str(HERE))

import document_rag as desk

# Reuses Chroma when the hash of parsed/elements.jsonl still matches.
# The first build embeds a few hundred chunks and takes several minutes.
chunks = desk.build_index(force=False, caption=True)
print(len(chunks), "chunks")


Index already matches the parsed files (284 chunks).
284 chunks


## What goes into a chunk

| Element | What the index stores | Why |
|---|---|---|
| `text` | About 900 characters per chunk, with 120 characters of overlap | A sentence near a cut can still appear with its context |
| `table` | One Markdown grid | Column headings and values stay together |
| `figure` | The whole caption, with the page image path in metadata | Search can match the caption and you can open the image to verify it |

Each chunk starts with the document ID, file type, page number, and element type. Keyword and vector search both use this text.


In [2]:
shown = set()
for chunk in chunks:
    kind = chunk.metadata["element_type"]
    if kind in shown:
        continue
    shown.add(kind)
    print("=" * 60)
    print(chunk.metadata["chunk_id"])
    print(chunk.page_content[:450])
    print()
    if shown == {"text", "table", "figure"}:
        break


0082969-p1-text-1-1
Document 0082969 (pdf), page 1, text. – FINANCING ALTERNATIVE IN TIMES OF.
DEBT FUNDS 
– FINANCING ALTERNATIVE IN TIMES OF 
AMPLE MONETARY ACCOMMODATION 
 
 
 
 
 
Financial Research 
September 2018

1295982-p1-table-14-1
Document 1295982 (pdf), page 1, table. Portfolio Manager.
| Portfolio Manager | Hamish Douglass |
| --- | --- |
| Strategy Inception Date | 1 July 2013 |
| Total Global Equity Assets1 | USD $25,109.9 million |
| Total Strategy Assets | USD $2,726.6 million |

walker_ranch_seismic-p1-figure-70-1
Document walker_ranch_seismic (docx), page 1, figure. Walker Ranch seismic images.
Embedded figure in walker_ranch_seismic.docx. Use the paragraphs in this same file for the caption. Do not invent labels for the picture.



## Compare keyword and vector search

Search for `18,443.87` and check whether keyword search finds the Newton minutes. Then try the question about the Commercial Real Estate loss rate and compare the two rankings.

Reciprocal rank fusion adds `1 / (60 + r)` to a chunk's score for each list where it appears at rank `r`. A chunk that appears in both lists gets points from both. This is the same calculation used in Module 15.

`words()` indexes both `18,443.87` and `18443.87`, so you can try the amount with or without the comma.


In [3]:
def labels(docs):
    return [f"{d.metadata['doc_id']} p.{d.metadata['page']} {d.metadata['element_type']}" for d in docs]

amount = "18,443.87"
print("keyword:", labels(desk.keyword_search(amount, k=3)))
print("vector: ", labels(desk.vector_search(amount, k=3)))
print("fused:  ", labels(desk.hybrid_search(amount, k=4)))
print()
vague = "commercial real estate portfolio loss rate under the adverse scenario"
print("keyword:", labels(desk.keyword_search(vague, k=3)))
print("vector: ", labels(desk.vector_search(vague, k=3)))
print("fused:  ", labels(desk.hybrid_search(vague, k=4)))


keyword: ['2025833 p.6 text']


vector:  ['2025833 p.4 text', '2025833 p.6 text', '2025833 p.7 text']
fused:   ['2025833 p.4 text', '2025833 p.6 text', '2025833 p.6 text', '2025833 p.7 text']

keyword: ['7257900 p.3 text', '7257900 p.7 text', '7257900 p.7 figure']


vector:  ['7257900 p.7 text', '7257900 p.3 text', '7257900 p.7 figure']
fused:   ['7257900 p.7 text', '7257900 p.3 text', '7257900 p.7 figure', '7257900 p.1 text']


## Score the first four results

`retrieval_report` counts a document hit when the right source is in the first four chunks. It counts a page hit when those chunks include a page listed as evidence for the question.

The saved run got 16/16 on both checks. Run the report again and look at any `MISS` rows. Start by checking whether search returned the page you needed. If it did, inspect the retrieved chunk to see whether it actually contains the requested fact.


In [4]:
rows = desk.retrieval_report(k=4)
doc_hits = sum(row["hit_doc"] for row in rows)
page_hits = sum(row["hit_page"] for row in rows)
print(f"doc hit {doc_hits}/{len(rows)}    page hit {page_hits}/{len(rows)}\n")
for row in rows:
    mark = "ok  " if row["hit_doc"] and row["hit_page"] else "MISS"
    print(f"{mark} {row['answer_type'][:22]:22} {row['doc_id']:22} {row['top']}")


doc hit 16/16    page hit 16/16

ok   table_required         6002966                ['6002966 p.10 text', '6002966 p.5 text', '6002966 p.3 text', '6002966 p.4 text']
ok   image_plus_text_as_ans 1295982                ['1295982 p.3 text', '1295982 p.3 text', '1295982 p.3 text', '1295982 p.3 figure']
ok   table_required         6838482                ['6838482 p.7 text', '2025833 p.3 text', '2025833 p.3 text', '6838482 p.1 text']
ok   table_required         2025833                ['2025833 p.4 text', '2025833 p.6 text', '2025833 p.6 text', '2025833 p.2 text']
ok   text_only              0082969                ['0082969 p.8 text', '0082969 p.3 text', '0082969 p.9 text', '0082969 p.7 text']
ok   image_plus_text_as_ans 1295982                ['1295982 p.3 text', '1295982 p.3 figure', '1295982 p.3 text', '1295982 p.3 text']
ok   table_required         2025833                ['2025833 p.4 text', '2025833 p.3 text', '2025833 p.6 text', '2025833 p.6 text']
ok   table_required         2025833   

## Exercise

Run `hybrid_search` on these two questions and write down the top document id for each.

1. `In the Walker Ranch seismic figures, what do the red dots show?`
2. `In PNC's adverse scenario, what is the Commercial Real Estate portfolio loss rate?`

<details><summary>Answer</summary>

1. `walker_ranch_seismic`. The geothermal note is in the packet so a finance question is not the only thing the index contains. This question should not cite a shareholder letter.
2. `7257900`, and the figure chunk for page 7 should be in the list. That chunk is the checked Table 5 transcription. The prose chunk on the same page talks about CRE as a share of total losses, which is a different number.

</details>


In [5]:
for question in [
    "In the Walker Ranch seismic figures, what do the red dots show?",
    "In PNC's adverse scenario, what is the Commercial Real Estate portfolio loss rate?",
]:
    print(question)
    print(" ", labels(desk.hybrid_search(question, k=4)))
    print()


In the Walker Ranch seismic figures, what do the red dots show?
  ['walker_ranch_seismic p.1 text', 'walker_ranch_seismic p.1 text', 'walker_ranch_seismic p.1 figure', 'walker_ranch_seismic p.1 text']

In PNC's adverse scenario, what is the Commercial Real Estate portfolio loss rate?


  ['7257900 p.7 text', '7257900 p.3 text', '7257900 p.7 figure', '7257900 p.1 text']



## When the same page appears more than once

The first four results can include several chunks from one page. Look at the overlapping text to see whether the subject and the number appear together in any of those chunks.

In [Notebook 3](03_sign_the_draft.ipynb), the model drafts from the retrieved text. You'll open the page image to check the answer before approving it.
